# Phase 5.1b: Simple 1D-CNN v2

**Diagnostics from Phase 5.1:**
- Model overfit by epoch 3 (val_loss climbing while train_loss dropping).
- Model predicted "Stress" ~61% of the time, but actual test Stress rate is 40%. The model learned the Train prior (56% Stress), not actual discrimination.
- Test accuracy 0.483 < majority baseline 0.604. (model is below baseline)

*Three changes:*
1. **Class Weights in CrossEntropyLoss (inverse frequency)**: Removes the model's incentive to default to the majority train class.
2. **Smaller Model**: 4 temporal filters, 8 spatial filters (~3K params vs 8K). Less capacity to memorize the 1920-segment train set.
3. **Stronger Regularization**: dropout 0.6 (was 0.5), weight_decay 1e-3 (10x), lower lr 5e-4 (slows learning so val gets monitored more carefully).
 
If this works, the pipeline is sound and EEGNet should do at least as well.
If it doesn't help, the architecture is the bottleneck and EEGNet's depthwise design should do better.

In [1]:

import os
import json
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
 
from phase5_utils import (
    load_phase2_data, make_subject_split, make_loaders,
    train_model, evaluate_model, count_parameters,
    PHASE5_DIR, SEED,
)
 
torch.manual_seed(SEED)
np.random.seed(SEED)

# Architecture

Same structure as Phase 5.1 but smaller capacity

In [2]:

"""
Smaller version of SimpleEEGCNN. Same temporal → spatial → temporal pattern,
but with fewer filters and higher dropout.

Input:  (B, 1, 32, 640)
Output: (B, n_classes)
"""

class SimpleEEGCNNv2(nn.Module):
 
    def __init__(self, n_classes=2, n_temporal=4, n_spatial=8, dropout=0.6):
        super().__init__()
 
        # Block 1: temporal conv (only 4 filters now) --------
        self.temporal_conv = nn.Conv2d(
            1, n_temporal, kernel_size=(1, 25), padding=(0, 12), bias=False
        )
        self.bn1 = nn.BatchNorm2d(n_temporal)
 
        # Block 2: spatial conv (only 8 output filters now) --------
        self.spatial_conv = nn.Conv2d(
            n_temporal, n_spatial, kernel_size=(32, 1), bias=False
        )
        self.bn2 = nn.BatchNorm2d(n_spatial)
        self.pool1 = nn.AvgPool2d(kernel_size=(1, 4))
        self.drop1 = nn.Dropout(dropout)
 
        # Block 3: deeper temporal conv --------
        self.temporal_conv2 = nn.Conv2d(
            n_spatial, n_spatial, kernel_size=(1, 13), padding=(0, 6), bias=False
        )
        self.bn3 = nn.BatchNorm2d(n_spatial)
        self.pool2 = nn.AvgPool2d(kernel_size=(1, 8))
        self.drop2 = nn.Dropout(dropout)
 
        # Classifier --------
        self.classifier = nn.Linear(n_spatial * 20, n_classes)
 
    def forward(self, x):
        x = self.temporal_conv(x)
        x = self.bn1(x)
 
        x = self.spatial_conv(x)
        x = self.bn2(x)
        x = F.elu(x)
        x = self.pool1(x)
        x = self.drop1(x)
 
        x = self.temporal_conv2(x)
        x = self.bn3(x)
        x = F.elu(x)
        x = self.pool2(x)
        x = self.drop2(x)
 
        x = x.flatten(1)
        x = self.classifier(x)
        return x

# Main

In [3]:
print("PHASE 5.1b: SIMPLE 1D-CNN v2 (class weights + stronger regularization)")
print("=" * 70)
 
 
# 1. Load data + split --------

print("\n[1/5] Loading data and recreating Phase 5.0 split...")
X, y_binary, subjects = load_phase2_data(verbose=False)
train_idx, val_idx, test_idx, split_info = make_subject_split(subjects, n_train=32, n_val=4, n_test=4, seed=SEED)

train_loader, val_loader, test_loader = make_loaders(X, y_binary, train_idx, val_idx, test_idx, batch_size=64)

print(f"  Train: {train_idx.sum()}  Val: {val_idx.sum()}  Test: {test_idx.sum()}")


# 2. Compute class weights from Train labels only --------
"""
Inverse-frequency weighting: rare class gets a higher weight in the loss.
Formula: w_c = n_total / (n_classes * n_in_class_c)
This makes the per-class total weight equal, so the optimizer can't get away
with always predicting the majority class.
"""

print("\n[2/5] Computing inverse-frequency class weights from TRAIN set...")
y_train = y_binary[train_idx]
class_counts = np.bincount(y_train, minlength=2)
n_total = class_counts.sum()
n_classes = 2
class_weights_np = n_total / (n_classes * class_counts)
class_weights = torch.FloatTensor(class_weights_np)
print(f"  Train class counts: Relaxed={class_counts[0]}, Stress={class_counts[1]}")
print(f"  Class weights: Relaxed={class_weights_np[0]:.3f}, Stress={class_weights_np[1]:.3f}")
print(f"  -> Relaxed gets a higher weight because it's the minority class in train.")

# Build the weighted loss function
criterion = nn.CrossEntropyLoss(weight=class_weights)


# 3. Build model --------
print("\n[3/5] Building SimpleEEGCNNv2 (smaller, more dropout)...")
device = "cpu"
model = SimpleEEGCNNv2(n_classes=2, n_temporal=4, n_spatial=8, dropout=0.6).to(device)
n_params = count_parameters(model)
print(f"  Total trainable parameters: {n_params:,}  (was 8,346 in 5.1)")
 
with torch.no_grad():
    dummy = torch.zeros(2, 1, 32, 640)
    out = model(dummy)
    print(f"  Forward-pass sanity: input {tuple(dummy.shape)} → output {tuple(out.shape)}")


# 4. Train ---------
print("\n[4/5] Training with class-weighted loss...")
start = time.time()
best_state, history = train_model(
    model, train_loader, val_loader,
    n_epochs=60, lr=5e-4, weight_decay=1e-3, patience=10,
    device=device, verbose=True,
    criterion=criterion,   # <-- weighted CE loss
)
elapsed = time.time() - start
print(f"  Training time: {elapsed:.1f} seconds ({elapsed/60:.1f} min)")

model.load_state_dict(best_state)
 
 
# 5. Evaluate --------
print("\n[5/5] Evaluating on test set...")
results = evaluate_model(model, test_loader, device=device)
 
n0_test = int((y_binary[test_idx] == 0).sum())
n1_test = int((y_binary[test_idx] == 1).sum())
majority_baseline = max(n0_test, n1_test) / (n0_test + n1_test)
 
# Also compute the model's actual class-prediction rate, so we can verify theclass-weight fix worked
pred_counts = np.bincount(results["preds"], minlength=2)
pred_rate_stress = pred_counts[1] / pred_counts.sum()
 
print(f"  Test accuracy : {results['accuracy']:.4f}")
print(f"  Test F1 (pos) : {results['f1']:.4f}")
print(f"  Test F1 macro : {results['f1_macro']:.4f}")
print(f"  Confusion matrix:")
print(f"    {results['confusion_matrix']}")
print(f"  ---- Diagnostic checks ----")
print(f"  Model's prediction rate of 'Stress': {pred_rate_stress:.3f}")
print(f"    (Train prior was 0.560, true test rate is 0.396 — closer to 0.4 = good)")
print(f"  ---- Reference points ----")
print(f"  Majority-class baseline on test  : {majority_baseline:.4f}")
print(f"  Phase 4B best (binary RF, LOSO)  : 0.5870")
print(f"  Phase 5.1 (no class weights)     : 0.4833")

PHASE 5.1b: SIMPLE 1D-CNN v2 (class weights + stronger regularization)

[1/5] Loading data and recreating Phase 5.0 split...
  Train: 1920  Val: 240  Test: 240

[2/5] Computing inverse-frequency class weights from TRAIN set...
  Train class counts: Relaxed=845, Stress=1075
  Class weights: Relaxed=1.136, Stress=0.893
  -> Relaxed gets a higher weight because it's the minority class in train.

[3/5] Building SimpleEEGCNNv2 (smaller, more dropout)...
  Total trainable parameters: 2,318  (was 8,346 in 5.1)
  Forward-pass sanity: input (2, 1, 32, 640) → output (2, 2)

[4/5] Training with class-weighted loss...
  Epoch   1 | train_loss 0.7092 | val_loss 0.6933 | val_acc 0.5250
  Epoch   2 | train_loss 0.7012 | val_loss 0.6912 | val_acc 0.5583
  Epoch   3 | train_loss 0.6963 | val_loss 0.6903 | val_acc 0.5417
  Epoch   4 | train_loss 0.6998 | val_loss 0.6874 | val_acc 0.5750
  Epoch   5 | train_loss 0.6906 | val_loss 0.6895 | val_acc 0.5708
  Epoch   6 | train_loss 0.6861 | val_loss 0.6899 |

# Save Results and Plots

In [4]:
out_dir = os.path.join(PHASE5_DIR, "phase5_1b_simple_cnn_v2")
os.makedirs(out_dir, exist_ok=True)
 
# Training curves --------
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
epochs = range(1, len(history["train_loss"]) + 1)
axes[0].plot(epochs, history["train_loss"], label="Train", color="#5DA5DA", marker="o", markersize=3)
axes[0].plot(epochs, history["val_loss"],   label="Val",   color="#F15854", marker="o", markersize=3)
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Weighted CE loss")
axes[0].set_title("Loss curves (class-weighted)"); axes[0].legend(); axes[0].grid(alpha=0.3)
 
axes[1].plot(epochs, history["val_acc"], color="#60BD68", marker="o", markersize=3)
axes[1].axhline(majority_baseline, color="gray", linestyle="--", label=f"Majority baseline ({majority_baseline:.3f})")
axes[1].axhline(0.5870, color="purple", linestyle=":", label="Phase 4B RF (0.587)")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Validation accuracy")
axes[1].set_title("Validation accuracy")
axes[1].legend(); axes[1].grid(alpha=0.3); axes[1].set_ylim(0, 1)
 
plt.suptitle(f"Phase 5.1b — Simple 1D-CNN v2 ({n_params:,} params)")
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "training_curves.png"), dpi=150)
plt.close()
 
# Confusion matrix --------
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(results["confusion_matrix"], annot=True, fmt="d", cmap="Blues",xticklabels=["Relaxed", "Stress"], yticklabels=["Relaxed", "Stress"], ax=ax,
)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"Phase 5.1b test confusion matrix (acc={results['accuracy']:.3f}, "f"F1-macro={results['f1_macro']:.3f})")
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "confusion_matrix.png"), dpi=150)
plt.close()
 
# JSON summary --------
results_summary = {
    "model": "SimpleEEGCNNv2",
    "n_params": int(n_params),
    "training_time_sec": float(elapsed),
    "n_epochs_trained": len(history["train_loss"]),
    "test_accuracy": float(results["accuracy"]),
    "test_f1": float(results["f1"]),
    "test_f1_macro": float(results["f1_macro"]),
    "majority_baseline": float(majority_baseline),
    "predicted_stress_rate": float(pred_rate_stress),
    "actual_test_stress_rate": float(n1_test / (n0_test + n1_test)),
    "train_stress_rate": float(class_counts[1] / n_total),
    "class_weights": class_weights_np.tolist(),
    "best_val_loss": float(min(history["val_loss"])),
    "best_val_acc": float(max(history["val_acc"])),
    "confusion_matrix": results["confusion_matrix"].tolist(),
    "split_subjects": split_info,
    "hyperparameters": {
        "n_temporal_filters": 4,
        "n_spatial_filters": 8,
        "dropout": 0.6,
        "lr": 5e-4,
        "weight_decay": 1e-3,
        "max_epochs": 60,
        "patience": 10,
        "class_weighted_loss": True,
    },
}
with open(os.path.join(out_dir, "results.json"), "w") as f:
    json.dump(results_summary, f, indent=2)
 
torch.save(best_state, os.path.join(out_dir, "simple_cnn_v2_best.pt"))
 
print(f"\nSaved outputs to: {out_dir}")
print("=" * 70)
print("Phase 5.1b complete.")
print("=" * 70)


Saved outputs to: C:\Users\hibro\OneDrive\Desktop\Desktop_Files\Projects\Python\ML_Models\Cognitive_Stress_Classification\EEG-Stress-Classification\Results\phase5\phase5_1b_simple_cnn_v2
Phase 5.1b complete.
